In [ ]:
from binance.client import Client
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import datetime
import json
import boto3
client = Client('YOUR_BINANCE_API_KEY', 'YOUR_BINANCE_API_SECRET')
#client = Client('YOUR_BINANCE_API_KEY_2', 'YOUR_BINANCE_API_SECRET_2')

In [ ]:
sd = '8-Jul-2021 01:00'
ts = int(datetime.datetime.strptime(sd, '%d-%b-%Y %H:%M').timestamp())*1000
#ts = 1625097600000
#ts=None
trades = []
while True:
    ct = client.futures_account_trades(startTime=ts, limit=1000)
    if ct:
        trades.extend(ct)
        ts = ct[-1]['time']+1
        print(ts)
    else:
        break
trades = [x for x in trades if x['symbol'] != 'DGBUSDT']
df = pd.DataFrame(trades).set_index('time')
df.index = pd.to_datetime(df.index, unit='ms')
#df = df.loc[sd:, :]
df['realizedPnl'] = df['realizedPnl'].astype(float)
rpnl = df.loc[:, ['realizedPnl', 'symbol']].groupby('symbol').sum().astype(float)
rpnl = df.loc[:, 'realizedPnl']
plt.figure()
rpnl_1h = rpnl.resample('1H').sum()
rpnl_1h.cumsum().plot(marker='.', figsize=(15,5))
plt.grid(True)
plt.grid(True, which='minor')
print('total pnl', rpnl.sum())
print('sharpe', rpnl_1h.mean()/rpnl_1h.std())
print('last', rpnl_1h.index[0], rpnl_1h.index[-1])

In [ ]:
ssm = boto3.client('ssm', region_name='eu-west-1')
res = ssm.get_parameter(Name='/binance/pairs_v2/state')
state = json.loads(res['Parameter']['Value'])
aa=client.futures_position_information()
upnl = {x['symbol']: float(x['unRealizedProfit']) for x in aa}
pairs = [tuple(s['pair']) for s in state]
pair_pnl = pd.Series([upnl[p1]+upnl[p2] for p1, p2 in pairs], index=[f'{p1}_{p2}' for p1, p2 in pairs])
pair_pnl.plot(kind='bar')
pair_pnl.sum()
aa=client.futures_account()
plt.title(float(aa['totalMaintMargin'])/float(aa['totalMarginBalance'])*100)

In [ ]:
import numpy as np
def proc_log(line):
    ret = (np.array(line['final_price'])-np.array(line['initial_price']))/np.array(line['initial_price'])
    ret = (ret[0]-ret[1])/2*(1 if line['state'] == 'LONG' else -1)
    return pd.Series([ret, line['initial_ratio'], datetime.datetime.strptime(line.get('date_added', '1-Jan-1970 00:00'), '%d-%b-%Y %H:%M'), datetime.datetime.strptime(line.get('date_closed', '1-Jan-1970 00:00'), '%d-%b-%Y %H:%M'), '_'.join(line['pair'])], index=['return', 'ratio', 'date_added', 'date_closed', 'pair'])
start_str = 'INFO - Closing trade'
with open('log') as f:
    lines = [x for x in f.readlines() if x.startswith(start_str)]
print(len(lines))
lines = [json.loads(x.replace(start_str, '').strip()) for x in lines]
lines = [x for x in lines if 'initial_price' in x and 'final_price' in x]
print(len(lines))
data = pd.concat([proc_log(x) for x in lines], axis=1).transpose().sort_values(['date_closed', 'date_added'])
plt.scatter(data['ratio'].abs(), data['return'])
plt.axhline(0, color='k')
plt.axvline(0, color='k')
plt.xlabel('ratio')
plt.ylabel('return')
data

In [ ]:
with open('log_account') as f:
    lines = [json.loads(x) for x in f]
mb = pd.Series([float(x['marginBalance'])+float(x.get('bnb', 0)) for x in lines], index=[datetime.datetime.strptime(x['snapshotTime'], '%d-%b-%Y %H:%M') for x in lines])
mb.plot(marker='.')
wb = pd.Series([float(x['walletBalance'])+float(x.get('bnb', 0)) for x in lines], index=[datetime.datetime.strptime(x['snapshotTime'], '%d-%b-%Y %H:%M') for x in lines])
wb.plot(marker='.', figsize=(15,5))
plt.grid(True)
plt.figure()
mbd = mb.diff(1)
mbd[mbd>200] = np.nan
mbd.plot(figsize=(15,5))
plt.axhline(0, color='k')
plt.grid(True, which='minor')

In [ ]:
client = Client('YOUR_BINANCE_API_KEY_2', 'YOUR_BINANCE_API_SECRET_2')
aa=client.futures_account()
print(float(aa['totalMaintMargin'])/float(aa['totalMarginBalance'])*100)
client.futures_account()['assets']

In [ ]:
to_convert_usdt = 100
client.futures_account_transfer(asset='USDT', amount=to_convert_usdt, type=2)
client.create_order(symbol='BNBUSDT', quoteOrderQty=to_convert_usdt, type='MARKET', side='BUY')
qty_bnb = float(client.get_asset_balance(asset='BNB')['free'])
print('bnb found', qty_bnb)
client.futures_account_transfer(asset='BNB', amount=qty_bnb, type=1)

In [ ]:
client.futures_account()['assets']

In [ ]:
client.futures_account()

In [ ]:
round(12, 0)